# Evaluating Cost-Sensitive Explainable Ensembles Against Deep Learning for Proactive Workforce Retention
**Group 6 - 3CSD - Final Project**

**Objective.** Train attrition models on the *Employee Attrition Classification Dataset*, then test cross-domain on the *HR Analytics Dataset* to expose the recall gap that standard accuracy hides. Apply a false-negative cost matrix, prove the optimization recovers recall, then explain the optimized ensembles with SHAP and DiCE.

**Pipeline.** Data Setup → Initial Training (3 Models) → Benchmark → Cost Matrix → Optimized Retraining → Explainability (SOTA Model) → Prescriptive Export.

In [ ]:
import os, re, random, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, confusion_matrix, classification_report)

import xgboost as xgb

#DNN
import tensorflow as tf 
from tensorflow.keras import layers, models as kmodels, callbacks

import shap
import dice_ml
import joblib

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=UserWarning)
pd.set_option('display.max_columns', 60)
sns.set_style('whitegrid')

print(f'TensorFlow {tf.__version__} | XGBoost {xgb.__version__} | SHAP {shap.__version__} | DiCE {dice_ml.__version__}')

## Step 1 — Data Setup
Load both datasets, normalize their schemas, perform **feature intersection**, then encode categoricals and split into train / validation / test.

In [ ]:
DATA_DIR   = './data'
TRAIN_PATH = os.path.join(DATA_DIR, 'training_validation', 'Stealth_Attrition.csv')
TEST_PATH  = os.path.join(DATA_DIR, 'test', 'HR_Analytics.csv')

assert os.path.exists(TRAIN_PATH), f'Missing training file: {TRAIN_PATH}'
assert os.path.exists(TEST_PATH),  f'Missing testing file:  {TEST_PATH}'

df_train_raw = pd.read_csv(TRAIN_PATH)
df_test_raw  = pd.read_csv(TEST_PATH)

print('Stealth (train):    ', df_train_raw.shape)
print('HR Analytics (test):', df_test_raw.shape)
print('\nStealth columns:\n ', list(df_train_raw.columns))
print('\nHR Analytics columns:\n ', list(df_test_raw.columns))

## Step 2 — Initial Training (Three Branches)
Train three model branches on the Stealth dataset using **default loss functions** (no cost weighting yet). This baseline is what Step 3 will use to demonstrate the recall failure.

- **Branch A — Baseline:** Logistic Regression, CARTs
- **Branch B — Deep Neural Network:** TensorFlow / Keras
- **Branch C — Ensembles:** Random Forest, XGBoost

## Step 3 — Benchmark
Apply every Step-2 model to the HR Analytics test set and report **accuracy, precision, recall, and F1** plus a confusion-matrix grid.

## Step 4 — Cost Matrix
Define a custom cost matrix that mathematically penalizes the model **X times more for False Negatives** (missing an employee who quits) than for False Positives (false alarm).

## Step 5 — Optimized Retraining
Refit every branch with the penalty wired into its native mechanism, then compare the new metrics against Step 3 to prove the optimization worked.

## Step 6 — Explainability (SHAP & DiCE)
### SOTA MODEL ONLY
Apply explainability **only to the optimized ensembles** (Random Forest + XGBoost).

- **SHAP:** global feature importance (bar + beeswarm) and local explanations (waterfall) for the highest-risk employees.
- **DiCE:** counterfactual recommendations - *what minimal feature changes would flip a high-risk employee's prediction back to “stay”?*

## Step 7 — Final Output